In [ ]:
# ==============================================================================
# STEP 1: VISUALIZING PHASE 02 (Raw Skeleton Proof)
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
import os
import random
from google.colab import drive

# 1. MOUNT DRIVE & CORRECT PATHS
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/Human-Fall-Recognition"
RAW_SKELETONS_DIR = os.path.join(BASE_DIR, "Skeleton_Raw") # FIXED PATH
FIGURES_DIR = os.path.join(BASE_DIR, "Results/Figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# 2. DEFINE SKELETON EDGES (COCO Format)
EDGES = [(0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
         (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]

# 3. PICK A SAMPLE FROM "Slip" (Or any class)
sample_class = "Slip"
class_path = os.path.join(RAW_SKELETONS_DIR, sample_class)
sample_files = [f for f in os.listdir(class_path) if f.endswith('.npy')]
skeleton_data = np.load(os.path.join(class_path, random.choice(sample_files)))

# 4. PLOT A SINGLE FRAME (Mid-sequence)
frame_idx = len(skeleton_data) // 2
keypoints = skeleton_data[frame_idx] # (17, 3) -> x, y, conf

plt.figure(figsize=(6, 8))
plt.title(f"Figure 3.X: Skeleton Keypoint Extraction (Phase 2 - {sample_class})")

# Draw Connections
for edge in EDGES:
    pt1, pt2 = keypoints[edge[0]], keypoints[edge[1]]
    if pt1[2] > 0.3 and pt2[2] > 0.3: # Confidence threshold
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], color='blue', linewidth=2, marker='o', mfc='red')

plt.gca().invert_yaxis()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(FIGURES_DIR, "phase_02_skeleton_proof.png"), dpi=300)
plt.show()

In [ ]:
# ==============================================================================
# STEP 2: VISUALIZING PHASE 03 (Sequence & Motion Analysis)
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. CORRECT PATHS
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "Sequences") # FIXED PATH
X = np.load(os.path.join(PROCESSED_DATA_DIR, "X.npy"))
y = np.load(os.path.join(PROCESSED_DATA_DIR, "y.npy"))
LABEL_MAP = {0: 'NoFall', 1: 'Faint', 2: 'Slip', 3: 'Trip'}

# 2. CLASS DISTRIBUTION BAR CHART
plt.figure(figsize=(8, 5))
counts = [np.sum(y == i) for i in range(4)]
labels = [LABEL_MAP[i] for i in range(4)]
sns.barplot(x=labels, y=counts, palette="viridis")
plt.title("Figure 3.Y: Processed Sequence Distribution (Phase 3)")
plt.ylabel("Count of 50-Frame Windows")
plt.savefig(os.path.join(FIGURES_DIR, "phase_03_distribution.png"), dpi=300)
plt.show()

# 3. MOTION PATTERN (Head Y-coordinate)
# Correct Reshape for your data: (N, 50, 51) -> (N, 50, 17, 3)
X_reshaped = X.reshape(-1, 50, 17, 3)

plt.figure(figsize=(10, 5))
for cls_idx in [0, 2]: # Compare NoFall (0) vs Slip (2)
    sample_idx = np.where(y == cls_idx)[0][0]
    motion = X_reshaped[sample_idx, :, 0, 1] # Joint 0 (Head), Index 1 (Y-coord)
    plt.plot(motion, label=f"Class: {LABEL_MAP[cls_idx]}", linewidth=2.5)

plt.title("Figure 4.X: Temporal Motion Pattern (Head Y-Coordinate)")
plt.xlabel("Sequence Frame Index")
plt.ylabel("Normalized Vertical Position")
plt.legend()
plt.savefig(os.path.join(FIGURES_DIR, "phase_03_motion_pattern.png"), dpi=300)
plt.show()